# Лабораторная работа № 2. Итерационные методы решения СЛАУ и спектральной задачи
**Основное задание: итерационные методы**
1. Сгенерировать симметричную квадратную вещественную матрицу и столбец свободных членов системы линейных алгебраических уравнений $Ax=b$ размерности $n = 6$.
2. Найти решение системы, применяя методы из лабораторной 1.
3. С точностью $\varepsilon_1 = 10^{−4}$ и $\varepsilon_2 = 10^{−12}$ по норме $\lVert \cdot \rVert_{\infty}$ найти решение системы, используя: а) метод Якоби; б) метод Зейделя.
4. Используя априорные оценки, предварительно вычислить количество необходимых итераций. Сравнить с фактическим количеством итераций, которое определяется апостериорными оценками.

**Основное задание: спектральная задача**

5. Найти наибольшее по модулю собственное число и соответствующий ему собственный вектор матрицы $A$, используя: а) степенной метод; б) метод скалярных произведений.
6. Найти наименьшее по модулю собственное число и соответствующий
собственный вектор, используя: а) метод сдвига; б) метод обратных итераций.
7. Найти собственные числа с помощью любой библиотеки линейной алгебры. Сравнить результаты.
8. Вычислить число обусловленности, используя полученные значения
собственных чисел. Вычислить число обусловленности, применяя методы из лабораторной 1. Сравнить результаты.

## Итерационные методы

In [41]:
import numpy as np
from numpy.linalg import norm
np.set_printoptions(linewidth=200)

rng = np.random.default_rng(1)

n = 6
lower_bound, upper_bound = -100, 100 # диапазон значений случайной матрицы.

### 2.

In [42]:
import lab1 # Функции из лабораторной №1.

### 3, 4.
**a)**

In [43]:
eps1 = 1e-4
eps2 = 1e-16

In [44]:
def jacobi_solve(A, b, x0=None, tol=1e-14, max_iter=1000):
    n = len(b)
    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    diag = np.diag(A)

    # Априорная оценка
    q = 0
    for i in range(n):
        row_sum = 0
        for j in range(n):
            if i != j:
                row_sum += abs(A[i, j] / diag[i])
        if row_sum > q:
            q = row_sum
    
    if q < 1:
        # Оценка ||x^1 - x^0||
        x_diff_norm = np.max(np.abs(b)) / np.min(np.abs(diag))
        
        num = np.log(tol * (1 - q) / x_diff_norm)
        denom = np.log(q)
        apriori_iter = int(np.ceil(abs(num / denom)))
    else:
        apriori_iter = np.inf
    
    D_inv = np.diag(1 / np.diag(A))  # Обратная диагональная матрица
    R = A - np.diag(diag)      # Матрица с нулевой диагональю
    
    for i in range(max_iter):
        x_new = D_inv @ (b - R @ x)
        if np.isnan(x_new).any():
            print("NaN encountered.")
            return x, i + 1, apriori_iter
        
        if norm(x_new - x) < tol:
            return x_new, i + 1, apriori_iter

        x = x_new
    return x, max_iter, apriori_iter

**b)**

In [45]:
def seidel_solve(A, b, x0=None, tol=1e-14, max_iter=1000):
    n = len(b)
    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    diag = np.diag(A)
    # Априорная оценка
    q_jacobi = 0
    for i in range(n):
        row_sum = 0
        for j in range(n):
            if i != j:
                row_sum += abs(A[i, j] / diag[i])
        if row_sum > q_jacobi:
            q_jacobi = row_sum
    
    if q_jacobi < 1:
        q = q_jacobi**2
        
        # Оценка первой итерации
        x_1 = np.zeros(n)
        for i in range(n):
            s = 0
            for j in range(i):
                s += A[i, j] * x_1[j]
            x_1[i] = (b[i] - s) / A[i, i]
        x_diff_norm = np.max(np.abs(x_1))
        
        num = np.log(tol * (1 - q) / x_diff_norm)
        denom = np.log(q)
        apriori_iter = int(np.ceil(abs(num / denom)))
    else:
        apriori_iter = np.inf
    
    # Expand A = L + D + U
    D = np.diag(diag)
    L = np.tril(A, -1)
    U = np.triu(A, 1)

    LD = L + D
    
    for i in range(max_iter):
        x_new = lab1.gauss_inverse(LD, pivot="column") @ (b - U@x)
        if np.isnan(x_new).any():
            print("NaN encountered.")
            return x, i + 1
        
        if norm(x_new - x) < tol:
            return x_new, i + 1

        x = x_new
    return x, max_iter

## Спектральная задача

### 5.
**a) Степенной метод**

In [46]:
def eigen_power(A, x0=None, tol=1e-10, max_iter=1000):
    rng = np.random.default_rng()
    
    if x0 is None:
        x = rng.random(A.shape[1])
    else:
        x = x0.copy()
    
    x = x / np.linalg.norm(x)
    lambda_old = 0.0
    
    for i in range(max_iter):
        x_new = A @ x
        lambda_new = norm(x_new)
        x = x_new / lambda_new
        
        if abs(lambda_new - lambda_old) < tol:
            # Определяем знак собственного числа
            sign = np.sign(x @ (A @ x))
            return sign * lambda_new, x, i
        
        lambda_old = lambda_new
    return lambda_new, x, max_iter

**b) Метод скалярных произведений**


In [47]:
def eigen_scalar(A, x0=None, tol=1e-10, max_iter=1000):
    rng = np.random.default_rng()
    
    if x0 is None:
        x = rng.random(A.shape[1])
    else:
        x = x0.copy()
    
    x = x / np.linalg.norm(x)
    lambda_old = 0.0
    
    for i in range(max_iter):
        x_new = A @ x
        lambda_new = x @ x_new  # скалярное произведение x · (A x)
        
        if abs(lambda_new - lambda_old) < tol:
            return lambda_new, x, i
        
        x = x_new / norm(x_new)
        lambda_old = lambda_new
    return lambda_new, x, max_iter

### 6.
**a) Метод сдвига**

In [48]:
def eigen_shift(A, x0=None, tol=1e-10, max_iter=1000):
    """
    Находит наименьшее по модулю собственное число методом сдвига (sigma=0)
    Фактически - степенной метод для A^(-1)
    """
    rng = np.random.default_rng()
    
    if x0 is None:
        x = rng.random(A.shape[1])
    else:
        x = x0.copy()
    
    x = x / norm(x)
    lambda_old = 0.0
    
    for i in range(max_iter):
        # Решаем A @ x_new = x (вместо явного обращения)
        x_new = np.linalg.solve(A, x)
        lambda_new = norm(x_new)
        x = x_new / lambda_new
        
        if abs(lambda_new - lambda_old) < tol:
            # Наименьшее собственное число = 1 / lambda_new
            eigenvalue_min = 1.0 / lambda_new
            sign = np.sign(x @ (A @ x))
            return sign * eigenvalue_min, x, i
        
        lambda_old = lambda_new
    eigenvalue_min = 1.0 / lambda_new
    return eigenvalue_min, x, max_iter

**b) Метод обратных итераций**

In [49]:
def eigen_inv(A, x0=None, mu=0.0, tol=1e-10, max_iter=1000):
    """
    Метод обратных итераций для поиска собственного числа, ближайшего к mu
    При mu=0 находит наименьшее по модулю собственное число
    """
    rng = np.random.default_rng()
    
    if x0 is None:
        x = rng.random(A.shape[1])
    else:
        x = x0.copy()
    
    x = x / np.linalg.norm(x)
    
    # Матрица со сдвигом
    A_shift = A - mu * np.eye(A.shape[0])
    lambda_old = 0.0
    
    for i in range(max_iter):
        # Решаем (A - μI) @ x_new = x
        x_new = np.linalg.solve(A_shift, x)
        lambda_new = norm(x_new)
        x = x_new / lambda_new
        
        if abs(lambda_new - lambda_old) < tol:
            # Собственное число, ближайшее к mu
            eigenvalue = mu + 1.0 / lambda_new
            return eigenvalue, x, i
        
        lambda_old = lambda_new
    eigenvalue = mu + 1.0 / lambda_new
    return eigenvalue, x, max_iter

### Собираем всё вместе для статистического анализа


In [50]:
eps = (1e-4, 1e-12)

for k in range(20):
    A = rng.normal(0, upper_bound, (n, n))
    A = (A + A.T) / 2 # Symmetrized matrix.
    for i in range(n):
        A[i, i] = A[i, i] * 10 # Делаем диагональ доминирующей.

    b = rng.normal(0, upper_bound, n)
    
    # Iteration part
    solutions = {}

    x = lab1.gauss_solve(A, b, pivot="column")
    # There's not much else to include for the gaussian method.
    solutions["gauss"] = ( np.linalg.norm((A @ x) - b), )
    
    for i in range(len(eps)):
        jacobi_solution = jacobi_solve(A, b, tol=eps[i])
        solutions[f"jacobi-eps{i+1}"] = jacobi_solve(A, b, tol=eps[i])
        solutions[f"seidel-eps{i+1}"] = seidel_solve(A, b, tol=eps[i])

    # Spectral part
    eigen_solutions = {}

    eigen_solutions["power"] = eigen_power(A)
    eigen_solutions["scalar"] = eigen_scalar(A)
    eigen_solutions["shift"] = eigen_shift(A)
    eigen_solutions["inverse"] = eigen_inv(A)
    eigen_solutions["numpy"] = np.linalg.eig(A)

/tmp/ipykernel_93664/544597820.py:34: RuntimeWarning: overflow encountered in matmul
  x_new = D_inv @ (b - R @ x)
/tmp/ipykernel_93664/544597820.py:34: RuntimeWarning: invalid value encountered in matmul
  x_new = D_inv @ (b - R @ x)
/tmp/ipykernel_93664/1240831227.py:45: RuntimeWarning: overflow encountered in matmul
  x_new = lab1.gauss_inverse(LD, pivot="column") @ (b - U@x)
/tmp/ipykernel_93664/1240831227.py:45: RuntimeWarning: invalid value encountered in matmul
  x_new = lab1.gauss_inverse(LD, pivot="column") @ (b - U@x)


NaN encountered.
NaN encountered.
NaN encountered.
NaN encountered.
NaN encountered.
NaN encountered.


Анализ в отдельном `.pdf` файле.